## Inspection des données

In [34]:
from pathlib import Path
import pandas as pd
from PIL import Image
import random

In [2]:
#vérifier root
print("Current working directory:\n", Path.cwd(),"\n")

Current working directory:
 C:\Users\thaim\Documents\mini-projects\2025-2026\mlprojects\pneumoniacnnclassification\notebooks 



In [3]:
#aller à root folder
import os
os.chdir("..")
print("Now in:", Path.cwd())

Now in: C:\Users\thaim\Documents\mini-projects\2025-2026\mlprojects\pneumoniacnnclassification


In [38]:
DATA_ROOT = Path("data/chest_xray")
OUT_DIR = Path("eda_outputs")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP = {"NORMAL":0, "PNEUMONIA":1}
SPLITS = ["train","val","test"]
IMG_EXTS = {".jpg",".jpeg",".png"}

In [5]:
#fonction pour charger les données en dataframe
def build_df(split:str):
    rows = []
    split_dir = DATA_ROOT / split
    for class_name, y in LABEL_MAP.items():
        class_dir = split_dir / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing folder: {class_dir}")
        for p in class_dir.rglob("*"): #rglob("*") recursively list everythin inside
            if p.is_file() and p.suffix.lower() in IMG_EXTS :
                rows.append({
                    "path":str(p),
                    "label":int(y),
                    "class_name":class_name,
                    "split":split,
                })
    df = pd.DataFrame(rows).sample(frac=1.0,random_state=42).reset_index(drop=True)
    return df
train_df = build_df(split="train")
train_df.head()

,path,label,class_name,split
0,data\chest_xray\train\PNEUMONIA\person1288_vir...,1,PNEUMONIA,train
1,data\chest_xray\train\NORMAL\NORMAL2-IM-0816-0...,0,NORMAL,train
2,data\chest_xray\train\PNEUMONIA\person61_bacte...,1,PNEUMONIA,train
3,data\chest_xray\train\PNEUMONIA\person722_viru...,1,PNEUMONIA,train
4,data\chest_xray\train\PNEUMONIA\person1141_vir...,1,PNEUMONIA,train


In [6]:
#charger pour tous split
data_df = {}
for s in SPLITS:
    data_df[s] = build_df(split=s)
    print("_"*30,s,"_"*30)
    print(data_df[s].head())

______________________________ train ______________________________
                                                path  label class_name  split
0  data\chest_xray\train\PNEUMONIA\person1288_vir...      1  PNEUMONIA  train
1  data\chest_xray\train\NORMAL\NORMAL2-IM-0816-0...      0     NORMAL  train
2  data\chest_xray\train\PNEUMONIA\person61_bacte...      1  PNEUMONIA  train
3  data\chest_xray\train\PNEUMONIA\person722_viru...      1  PNEUMONIA  train
4  data\chest_xray\train\PNEUMONIA\person1141_vir...      1  PNEUMONIA  train
______________________________ val ______________________________
                                                path  label class_name split
0  data\chest_xray\val\NORMAL\NORMAL2-IM-1427-000...      0     NORMAL   val
1  data\chest_xray\val\NORMAL\NORMAL2-IM-1430-000...      0     NORMAL   val
2  data\chest_xray\val\NORMAL\NORMAL2-IM-1438-000...      0     NORMAL   val
3  data\chest_xray\val\PNEUMONIA\person1952_bacte...      1  PNEUMONIA   val
4  data\chest

In [7]:
data_df.keys()

dict_keys(['train', 'val', 'test'])

In [8]:
#vérifier s'il y a d'image où on ne peut pas ouvrir
def find_corrupt_paths(df:pd.DataFrame):
    bad = []
    for p in df["path"]:
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception:
            bad.append(p)
    return bad

In [9]:
for k,df in data_df.items():
    bad = find_corrupt_paths(df=df)
    if len(bad) != 0 :
        print(f"Corrupted files in {k} : {len(bad)}")
    else :
        print(f"No corrupted files in {k}")

No corrupted files in train
No corrupted files in val
No corrupted files in test


In [19]:
# pourcentage de chaque split
totalDataSize = sum(len(v) for v in data_df.values())
for k,df in data_df.items():
    lenDf = len(df)
    print(f"Size percentage of {k} : {((len(df)/totalDataSize)*100):.2f}% ({lenDf})")

Size percentage of train : 89.07% (5216)
Size percentage of val : 0.27% (16)
Size percentage of test : 10.66% (624)


### Vérification de la qualité variable des images
- Largeur
- Hauteur
- Min
- Max
- Moyenne
- écart-type

In [33]:
# Vérification
def collect_sizes(df):
    shapes = []
    for p in df["path"]:
        try :
            with Image.open(p) as im:
                w, h = im.size
                mode = im.mode
                channels = len(im.getbands())
                shapes.append({"width": w,
                               "height": h, "mode": mode,
                               "channels": channels})
        except Exception :
            continue
    out = pd.DataFrame(shapes)
    return out

In [32]:
for k,df in data_df.items():
    shapes = collect_sizes(df)
    print("_"*30,k,"_"*30)
    print(shapes.describe())
    print("-"*15)
    print("Unique sizes :", shapes.drop_duplicates().shape[0])
    print("-"*15)
    print(shapes.value_counts().head(10))
    print("-"*15)

______________________________ train ______________________________
             width       height     channels
count  5216.000000  5216.000000  5216.000000
mean   1320.610813   968.074770     1.108512
std     355.298743   378.855691     0.453088
min     384.000000   127.000000     1.000000
25%    1056.000000   688.000000     1.000000
50%    1284.000000   888.000000     1.000000
75%    1552.000000  1187.750000     1.000000
max    2916.000000  2663.000000     3.000000
---------------
Unique sizes : 4366
---------------
width  height  mode  channels
1072   648     L     1           7
1080   728     L     1           6
976    672     L     1           5
992    592     L     1           5
1008   704     L     1           5
1064   760     L     1           5
1216   872     L     1           5
1008   680     L     1           5
1072   664     L     1           4
1176   784     L     1           4
Name: count, dtype: int64
---------------
______________________________ val __________________

In [39]:
# save dataframes
for k,df in data_df.items():
    (df).to_csv(OUT_DIR / f"{k}.csv", index=False)

#### Vérification Déséquilibre possible entre classes

In [44]:
for k,df in data_df.items():
    totalLen = len(df)
    mask_pneu = df["label"] == 1
    pneuLen = len(df[mask_pneu])
    normLen = len(df[~mask_pneu])
    print("_"*30,k,"_"*30)
    print(f"Pneumonie en pourcentage : {(pneuLen/totalLen)*100:.2f}% ({pneuLen})")
    print(f"Normal en pourcentage : {(normLen/totalLen)*100:.2f}% ({normLen})")

______________________________ train ______________________________
Pneumonie en pourcentage : 74.29% (3875)
Normal en pourcentage : 25.71% (1341)
______________________________ val ______________________________
Pneumonie en pourcentage : 50.00% (8)
Normal en pourcentage : 50.00% (8)
______________________________ test ______________________________
Pneumonie en pourcentage : 62.50% (390)
Normal en pourcentage : 37.50% (234)


#### Vérification de la data leakage (absence de duplication de noms de fichiers entre train/val/test)

In [48]:
filenames = []
for k, df in data_df.items():
    df["filename"] = df["path"].astype(str).apply(lambda p: Path(p).name)
    filenames = filenames + df["filename"].tolist()
print(f"Absence de duplication de noms de fichiers entre train/val/test : {len(filenames) == len(set(filenames))}")

Absence de duplication de noms de fichiers entre train/val/test : True


## Conclusion

L’analyse exploratoire confirme que le dataset est exploitable pour l’entraînement, avec plusieurs points clés à retenir.

### 1) Structure du dataset
- Total : **5 856 images**
- **Train** : 5 216 (**89,07%**)
- **Validation** : 16 (**0,27%**)
- **Test** : 624 (**10,66%**)

### 2) Qualité des données
- **Aucun fichier corrompu** détecté dans `train/val/test`.
- Les chemins et labels sont cohérents après indexation.

### 3) Variabilité des images
- Forte hétérogénéité de tailles (résolutions multiples).
- Images majoritairement en niveaux de gris (avec quelques cas en 3 canaux).
- Cela justifie un prétraitement standardisé (resize/crop/normalisation).

### 4) Déséquilibre des classes
- Train : **PNEUMONIA 74,29%** (3875) vs **NORMAL 25,71%** (1341)
- Val : **50/50** (8/8), mais échantillon très petit
- Test : **PNEUMONIA 62,5%** (390) vs **NORMAL 37,5%** (234)

### 5) Vérification de la data leakage (noms de fichiers)
- Vérification ajoutée : **absence de duplication de noms de fichiers entre `train/val/test`**.
- Résultat obtenu : `len(filenames) == len(set(filenames))` → **à renseigner avec votre sortie (`True`/`False`)**.
- Important : ce contrôle est utile, mais il ne garantit pas à lui seul l’absence de fuite au **niveau patient** (un même patient peut avoir des noms de fichiers différents).

### 6) Implications pour la suite
- Dataset globalement prêt pour la phase de modélisation.
- Points de vigilance : déséquilibre de classes, validation très petite, et contrôle leakage à compléter idéalement au niveau patient.
